# BioRydberg 2.0: Geometry-Aware Quantum Transport with Rydberg Atom Arrays

**Author:** Syed Marjan Ghazi  
**Contest:** Pasqal QoolQit Contest 2026  
**Research question:** Can programmable neutral-atom geometries and controls produce spatial excitation patterns resembling a simplified biological excitation network?

## Abstract
This notebook compares coherent single-excitation transport with a driven Rydberg analog program. It uses QoolQit to construct, compile, and locally emulate neutral-atom programs. A bounded classical search adjusts a three-atom geometry and constant laser controls. The comparison concerns normalized spatial profiles only: the Hamiltonians, excitation-number sectors, and physical meanings differ.

## 1. Motivation and scientific background
Biological pigment networks motivate small coherent hopping models in which one excitation moves between sites. Neutral-atom arrays provide a different many-body system: geometry sets Rydberg interactions, while laser amplitude and detuning drive occupation dynamics. The experiment tests whether constrained geometry and control choices can reduce a precisely defined spatial-pattern discrepancy. It does not test photosynthetic efficiency or quantum advantage.

In [1]:
from pathlib import Path
from importlib.metadata import version
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analysis import normalize_rows, split_profile_rmse
from src.biological_model import three_site_example, load_hamiltonian_csv, simulate_unitary_transport
from src.neutral_atom_mapping import couplings_to_qoolqit_coordinates
from src.optimization import Controls, evaluate
from src.qoolqit_simulation import run_occupation_dynamics
from src.visualization import apply_publication_style, plot_baseline_optimized, plot_interaction_matrix, plot_network, plot_raw_and_total, plot_site_populations

apply_publication_style()
print('Python:', sys.version.split()[0])
print('QoolQit:', version('qoolqit'))
assert version('qoolqit') == '1.4.0'

Python: 3.12.9
QoolQit: 1.4.0


## 2. The two models
### Biological reference Hamiltonian
In the single-excitation basis, $H_{bio}=\sum_i \epsilon_i |i\rangle\langle i|+\sum_{i\ne j}J_{ij}|i\rangle\langle j|$. Direct matrix exponentiation gives $P_i(t)=|\langle i|\psi(t)\rangle|^2$, with $\sum_iP_i(t)=1$.

### Rydberg Analog Model
QoolQit programs a register and drive for a Hamiltonian with global laser coupling, detuning, and diagonal pair interactions proportional to $n_i n_j/r_{ij}^6$. We initialize atom 1 in the Rydberg state so both experiments begin with a localized excitation. The drive can leave the single-excitation sector, so raw occupations need not sum to one.

### Important distinction
Biological hopping terms transfer amplitude between sites. Standard Rydberg van der Waals terms shift energies according to joint occupation. Normalized Rydberg occupations therefore define a spatial-profile diagnostic, not a biological probability or quantum-state fidelity.

## 3. Baseline biological validation

In [2]:
H_bio, bio_times, bio_populations = three_site_example()
assert np.allclose(H_bio, H_bio.T.conj())
probability_error = np.max(np.abs(bio_populations.sum(axis=1) - 1.0))
print('Hamiltonian:\n', H_bio)
print(f'Maximum probability-conservation error: {probability_error:.3e}')
fig = plot_site_populations(bio_times, bio_populations, 'Three-site biological excitation dynamics', 'Excitation probability')
plt.show()

Hamiltonian:
 [[ 0.    1.    0.25]
 [ 1.    0.2   0.7 ]
 [ 0.25  0.7  -0.1 ]]
Maximum probability-conservation error: 2.442e-15


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\1291851343.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. QoolQit register, interaction matrix, and local emulation
The coupling-informed spring layout uses biological coupling magnitudes only as an embedding heuristic. The closest atom pair is rescaled to unit distance, making the largest dimensionless $1/r^6$ interaction equal to one. The following cell executes a genuine QoolQit `QuantumProgram` through `LocalEmulator`.

In [3]:
couplings = H_bio.copy(); np.fill_diagonal(couplings, 0.0)
baseline_coordinates = couplings_to_qoolqit_coordinates(couplings, seed=42)
baseline_controls = Controls(duration=10.0, amplitude=0.8, detuning=0.0)
baseline_run, baseline_metrics = evaluate(bio_populations, baseline_coordinates, baseline_controls, n_times=25)
assert baseline_run.program.is_compiled
print('Coordinates:', baseline_coordinates)
print('QoolQit interaction matrix:\n', baseline_run.interaction_matrix)
print('Local-emulator seconds:', round(baseline_run.elapsed_seconds, 3))
fig = plot_network(baseline_coordinates, baseline_run.interaction_matrix, 'Coupling-informed three-atom register'); plt.show()
fig = plot_interaction_matrix(baseline_run.interaction_matrix, 'Rydberg interaction matrix'); plt.show()

Coordinates: [(-0.1125254454028904, 0.7941599166653182), (0.4744413700582561, -0.015451074828374736), (-0.3619159246553657, -0.7787088418369434)]
QoolQit interaction matrix:
 [[0.         1.         0.06130485]
 [1.         0.         0.47454682]
 [0.06130485 0.47454682 0.        ]]
Local-emulator seconds: 0.176


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\1593912242.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_network(baseline_coordinates, baseline_run.interaction_matrix, 'Coupling-informed three-atom register'); plt.show()


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\1593912242.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_interaction_matrix(baseline_run.interaction_matrix, 'Rydberg interaction matrix'); plt.show()


## 5. Raw occupation and normalization policy
We report raw $\langle n_i\rangle$ and their total before normalization. A time point enters the spatial comparison only when total Rydberg occupation exceeds $10^{-6}$. Undefined profiles remain `NaN`; the analysis never replaces them with meaningful zeros. The experiment records the number of excluded points.

In [4]:
baseline_spatial, valid_mask = normalize_rows(baseline_run.occupations, eps=1e-6, return_mask=True)
print('Total Rydberg occupation range:', baseline_run.occupations.sum(axis=1).min(), baseline_run.occupations.sum(axis=1).max())
print('Excluded time points:', int((~valid_mask).sum()))
print('Baseline metrics:', baseline_metrics)
fig = plot_raw_and_total(baseline_run.relative_times, baseline_run.occupations); plt.show()

Total Rydberg occupation range: 1.0 1.6291479202478376
Excluded time points: 0
Baseline metrics: {'rmse_all': 0.259494365737678, 'rmse_tuning': 0.2543357890082178, 'rmse_held_out': 0.2649695155145792, 'valid_time_points': 25, 'excluded_time_points': 0}


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\1479123128.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_raw_and_total(baseline_run.relative_times, baseline_run.occupations); plt.show()


## 6. Geometry and laser-control optimization
A bounded SciPy differential-evolution search moves the third vertex of a normalized triangle, enforcing unit minimum separation and a finite register extent. A second search tunes constant amplitude $\Omega$, detuning $\Delta$, and duration. Seeds 11 and 29 cover geometry; seeds 23 and 41 cover controls. Each search uses `maxiter=3`, `popsize=4`. Even valid time indices define the optimization objective, while odd indices remain held out. Every objective evaluation compiles and locally emulates a QoolQit program.

The stored results below come from `python experiments/05_contest_study.py --maxiter 3 --popsize 4 --n-times 25`. Set `RECOMPUTE_FULL_STUDY=True` to repeat the complete bounded search.

In [5]:
RECOMPUTE_FULL_STUDY = False
summary_path = ROOT / 'results' / 'study_summary.json'
if RECOMPUTE_FULL_STUDY or not summary_path.exists():
    from experiments import __path__ as _experiment_path
    import importlib.util
    spec = importlib.util.spec_from_file_location('contest_study', ROOT / 'experiments' / '05_contest_study.py')
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    summary = module.run_study(maxiter=3, popsize=4, n_times=25)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
metrics = pd.read_csv(ROOT / 'results' / 'tables' / 'study_metrics.csv')
metrics

,model,rmse_all,rmse_tuning,rmse_held_out,valid_time_points,excluded_time_points,amplitude,detuning,duration
0,baseline,0.259494,0.254336,0.264970,25,0,0.800000,0.000000,10.000000
1,geometry_optimized,0.239121,0.235149,0.243352,25,0,0.800000,0.000000,10.000000
2,geometry_and_controls_optimized,0.227569,0.222552,0.232883,25,0,1.159083,0.330952,7.129593
3,seven_site_limited,0.182243,0.179063,0.185755,17,0,1.159083,0.330952,7.129593


## 7. Baseline versus optimized result

In [6]:
optimized_coordinates = [tuple(x) for x in summary['geometry_optimized']['coordinates']]
c = summary['fully_optimized']['controls']
optimized_controls = Controls(duration=c['duration'], amplitude=c['amplitude'], detuning=c['detuning'])
optimized_run, optimized_metrics_check = evaluate(bio_populations, optimized_coordinates, optimized_controls, n_times=25)
optimized_spatial = normalize_rows(optimized_run.occupations)
baseline_held = baseline_metrics['rmse_held_out']
optimized_held = optimized_metrics_check['rmse_held_out']
improvement = 100 * (baseline_held - optimized_held) / baseline_held
print(f'Held-out RMSE: {baseline_held:.6f} -> {optimized_held:.6f} ({improvement:.1f}% reduction)')
print('Optimized controls:', optimized_controls)
print('Optimized coordinates:', optimized_coordinates)
fig = plot_baseline_optimized(bio_populations, baseline_spatial, optimized_spatial, baseline_held, optimized_held); plt.show()

Held-out RMSE: 0.264970 -> 0.232883 (12.1% reduction)
Optimized controls: Controls(duration=7.12959303311041, amplitude=1.159083308635532, detuning=0.3309520577793118)
Optimized coordinates: [(-0.9855383586533283, -0.23467193558523722), (0.13767702796377332, -0.23467193558523722), (0.8478613306895548, 0.4693438711704744)]


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\2536385103.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_baseline_optimized(bio_populations, baseline_spatial, optimized_spatial, baseline_held, optimized_held); plt.show()


The held-out improvement means that this constrained driven-Rydberg model better matches the chosen normalized site-distribution curves on samples excluded from the objective. It does not establish Hamiltonian equivalence, state fidelity, biological realism, or quantum advantage.

## 8. Seven-site extension
The repository's seven-site matrix is illustrative and not a validated FMO Hamiltonian. We ran one local-emulator benchmark using the optimized three-site controls and a coupling-informed layout. We did not optimize this register, so the result measures feasibility only and should not be compared as an equivalent optimization study.

In [7]:
seven_h = load_hamiltonian_csv(ROOT / 'data' / 'fmo_inspired_7site.csv')
seven_c = seven_h.copy(); np.fill_diagonal(seven_c, 0.0)
seven_bio = simulate_unitary_transport(seven_h, np.linspace(0, 12, 240), initial_site=0)
seven_coordinates = couplings_to_qoolqit_coordinates(seven_c, seed=42)
seven_run = run_occupation_dynamics(seven_coordinates, duration=optimized_controls.duration, amplitude=optimized_controls.amplitude, detuning=optimized_controls.detuning, n_times=17, initial_site=0)
seven_metrics = split_profile_rmse(seven_bio, seven_run.occupations)
print('Seven-site limited benchmark:', seven_metrics)
print('Emulator seconds:', round(seven_run.elapsed_seconds, 3))
fig = plot_raw_and_total(seven_run.relative_times, seven_run.occupations); fig.axes[0].set_title('Seven-site illustrative benchmark'); plt.show()

Seven-site limited benchmark: {'rmse_all': 0.18224271889999494, 'rmse_tuning': 0.1790632388706198, 'rmse_held_out': 0.18575459430103403, 'valid_time_points': 17, 'excluded_time_points': 0}
Emulator seconds: 0.196


C:\Users\ANC\AppData\Local\Temp\ipykernel_15100\649195365.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = plot_raw_and_total(seven_run.relative_times, seven_run.occupations); fig.axes[0].set_title('Seven-site illustrative benchmark'); plt.show()


## 9. Limitations and negative findings
- Geometry cannot convert diagonal Rydberg interactions into biological hopping terms.
- A global drive changes total Rydberg occupation and can populate several excitation sectors.
- Relative-time alignment discards a claim of common physical timescale.
- The optimization budget is intentionally small and the improvement is modest. Two seeds do not establish global optimality.
- The odd/even split checks interpolation-level generalization within one trajectory, not generalization to a new biological network.
- The seven-site data are illustrative; no literature FMO claim is made.
- No hardware or cloud QPU result appears in this submission.

## 10. Conclusion and future work
The small-scale experiment found a reproducible held-out RMSE reduction from 0.2650 to 0.2329 after geometry and control optimization. This supports the narrow hypothesis that programmable neutral-atom parameters can improve a chosen spatial-pattern metric. Future work should test local detuning controls, larger evaluation splits, robust optimization under noise, literature-sourced biological Hamiltonians, and eventual hardware execution.

## 11. Provenance and research integrity
**Previous foundation:** biological reference models, initial QoolQit experiments, a coupling-to-geometry heuristic, illustrative datasets, and the first comparison notebook.

**New contest work:** localized QoolQit initial-state alignment, explicit invalid-profile masking, bounded two-seed optimization, held-out metrics, seven-site feasibility benchmarking, automated tests, publication figures, this executed notebook, and the two-slide presentation.

## 12. References and attribution
1. [Pasqal QoolQit documentation](https://docs.pasqal.com/qoolqit/)
2. [QoolQit source repository](https://github.com/pasqal-io/qoolqit)
3. [QoolQit Contest requirements](https://www.pasqal.com/resources/the-qoolqit-contest-is-live/)
4. QoolQit 1.4.0 and its Pulser-based local emulator perform the neutral-atom execution. NumPy, SciPy, Matplotlib, NetworkX, and pandas support reference dynamics, optimization, visualization, layout, and tables.